# Phase 1E — CNN-LSTM Hybrid + LSTM | Window=50

**Changements clés vs notebooks précédents :**
-  Window size = **50** au lieu de 30

-  CNN-LSTM architecture optimisée fournie
-  LSTM amélioré pour comparaison
-  Wandb intégré

---
### Table des matières
1. Setup & Wandb
2. Chargement données brutes
3. Scaler + RUL
4. Fenêtres glissantes W=50
5. Train/Val Split
6. Fonctions utilitaires
7. CNN-LSTM Hybride same as sweep 1


## 1. Setup & Wandb

In [ ]:
!pip install wandb -q

import numpy as np
import pandas as pd
import json, os, joblib
import matplotlib.pyplot as plt
import wandb
from wandb.integration.keras import WandbMetricsLogger
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, Conv1D, Dense, Dropout,
    GlobalAveragePooling1D, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

wandb.login()

print(f'TensorFlow : {tf.__version__}')
print(f'GPU dispo  : {len(tf.config.list_physical_devices("GPU")) > 0}')

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chiraz (chiraz-u) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


TensorFlow : 2.19.0
GPU dispo  : True


## 2. Chargement données brutes

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


BASE_DIR    = '/content/drive/MyDrive/industrial-ai-platform'
MODEL_DIR   = f'{BASE_DIR}/phase1E_w50'
CONFIG_PATH = f'{BASE_DIR}/feature_config.json'

os.makedirs(MODEL_DIR, exist_ok=True)

# Chemins fichiers NASA
TRAIN_PATH = f'{BASE_DIR}/Data/cmapss/train_FD001.txt'
TEST_PATH  = f'{BASE_DIR}/Data/cmapss/test_FD001.txt'
RUL_PATH   = f'{BASE_DIR}/Data/cmapss/RUL_FD001.txt'

COLUMNS = [
    'unit_id', 'time_cycle',
    'op_setting_1', 'op_setting_2', 'op_setting_3',
    *[f'sensor_{i:02d}' for i in range(1, 22)]
]

df_train = pd.read_csv(TRAIN_PATH, sep=r'\s+', header=None, names=COLUMNS)
df_test  = pd.read_csv(TEST_PATH,  sep=r'\s+', header=None, names=COLUMNS)
df_rul   = pd.read_csv(RUL_PATH,   header=None, names=['RUL_true'])

# Charger config — feature set
with open(CONFIG_PATH) as f:
    config = json.load(f)

SENSORS_TO_KEEP = config['sensors_to_keep']
SENSORS_TO_DROP = config['sensors_to_drop']
RUL_CAP         = config['rul_cap']          # 125
SETTING_COLS    = ['op_setting_1', 'op_setting_2', 'op_setting_3']

# Window size = 50 dans ce notebook
WINDOW_SIZE = 50
N_FEATURES  = len(SENSORS_TO_KEEP)

print(f'Train : {df_train.shape}  |  Test : {df_test.shape}')
print(f'Window={WINDOW_SIZE}  Features={N_FEATURES}  RUL cap={RUL_CAP}')

Mounted at /content/drive
Train : (20631, 26)  |  Test : (13096, 26)
Window=50  Features=14  RUL cap=125


In [ ]:
# ── Étape 1 : Drop colonnes inutiles ──────────────────────────────────────
COLS_TO_DROP = SENSORS_TO_DROP + SETTING_COLS
df_train = df_train.drop(columns=[c for c in COLS_TO_DROP if c in df_train.columns])
df_test  = df_test.drop(columns=[c for c in COLS_TO_DROP if c in df_test.columns])

print(f'Features après drop : {SENSORS_TO_KEEP}')

# ── Étape 2 : Calcul RUL train ─────────────────────────────────────────────
max_cycles       = df_train.groupby('unit_id')['time_cycle'].max().rename('max_cycle')
df_train         = df_train.merge(max_cycles, on='unit_id')
df_train['RUL']  = (df_train['max_cycle'] - df_train['time_cycle']).clip(upper=RUL_CAP)
df_train         = df_train.drop(columns=['max_cycle'])

# ── Étape 3 : Calcul RUL test ──────────────────────────────────────────────
df_test  = df_test.sort_values(['unit_id', 'time_cycle'])
rul_true = df_rul['RUL_true'].values
test_units = df_test['unit_id'].unique()
rul_map  = {uid: rul_true[i] for i, uid in enumerate(sorted(test_units))}

# RUL pour chaque ligne du test
def assign_test_rul(group):
    uid       = group['unit_id'].iloc[0]
    final_rul = rul_map[uid]
    n         = len(group)
    group     = group.copy()
    group['RUL'] = (np.arange(n-1, -1, -1) + final_rul).clip(max=RUL_CAP)
    return group

df_test = df_test.groupby('unit_id', group_keys=False).apply(assign_test_rul)

print(f' RUL calculé — Train: {df_train["RUL"].describe().round(1).to_dict()}')

Features après drop : ['sensor_02', 'sensor_03', 'sensor_04', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']
 RUL calculé — Train: {'count': 20631.0, 'mean': 86.8, 'std': 41.7, 'min': 0.0, '25%': 51.0, '50%': 103.0, '75%': 125.0, 'max': 125.0}


In [ ]:
# ── Étape 4 : Scaler Global (MinMax) ──────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler
import joblib

scaler = MinMaxScaler(feature_range=(0, 1))

# Fit UNIQUEMENT sur le train
df_train_scaled = df_train.copy()
df_train_scaled[SENSORS_TO_KEEP] = scaler.fit_transform(df_train[SENSORS_TO_KEEP])

# Transform sur le test — même scaler, pas de re-fit
df_test_scaled = df_test.copy()
df_test_scaled[SENSORS_TO_KEEP] = scaler.transform(df_test[SENSORS_TO_KEEP])

# Sauvegarder le scaler
joblib.dump(scaler, f'{MODEL_DIR}/minmax_scaler.pkl')

print(' Scaler global appliqué')
print(f'Plage après normalisation (train) :')
print(df_train_scaled[SENSORS_TO_KEEP].agg(['min','max']).T.round(3).head(5))

 Scaler global appliqué
Plage après normalisation (train) :
           min  max
sensor_02  0.0  1.0
sensor_03  0.0  1.0
sensor_04  0.0  1.0
sensor_07  0.0  1.0
sensor_08  0.0  1.0


## 4. Fenêtres glissantes W=50

In [ ]:
def create_sequences(df, feature_cols, rul_col='RUL', window_size=50):
    """
    Fenêtres glissantes pour le train.
    Retourne X (N, W, F), y (N,), unit_ids (N,)
    """
    X_list, y_list, uid_list = [], [], []
    for uid, group in df.groupby('unit_id'):
        data   = group[feature_cols].values
        labels = group[rul_col].values
        T      = len(data)
        if T < window_size:
            pad    = np.repeat(data[[0]], window_size - T, axis=0)
            data   = np.vstack([pad, data])
            labels = np.concatenate([np.repeat(labels[0], window_size - T), labels])
            T      = window_size
        for t in range(T - window_size + 1):
            X_list.append(data[t:t+window_size])
            y_list.append(labels[t+window_size-1])
            uid_list.append(uid)
    return (np.array(X_list,   dtype=np.float32),
            np.array(y_list,   dtype=np.float32),
            np.array(uid_list, dtype=np.int32))


def create_test_sequences(df, feature_cols, rul_col='RUL', window_size=50):
    """
    Dernière fenêtre uniquement pour le test.
    """
    X_list, y_list = [], []
    for uid, group in df.groupby('unit_id'):
        data   = group[feature_cols].values
        labels = group[rul_col].values
        T      = len(data)
        if T < window_size:
            pad    = np.repeat(data[[0]], window_size - T, axis=0)
            data   = np.vstack([pad, data])
            labels = np.concatenate([np.repeat(labels[0], window_size - T), labels])
        X_list.append(data[-window_size:])
        y_list.append(labels[-1])
    return (np.array(X_list, dtype=np.float32),
            np.array(y_list, dtype=np.float32))


# Créer les séquences
X_train_full, y_train_full, unit_ids_full = create_sequences(
    df_train_scaled, SENSORS_TO_KEEP, window_size=WINDOW_SIZE
)
X_test, y_test = create_test_sequences(
    df_test_scaled, SENSORS_TO_KEEP, window_size=WINDOW_SIZE
)

# Sauvegarder unit_ids pour éviter de recalculer
np.save(f'{MODEL_DIR}/unit_ids_train_w50.npy', unit_ids_full)

print(f'X_train_full : {X_train_full.shape}')
print(f'y_train_full : {y_train_full.shape}')
print(f'X_test       : {X_test.shape}')
print(f'y_test       : {y_test.shape}')

X_train_full : (15731, 50, 14)
y_train_full : (15731,)
X_test       : (100, 50, 14)
y_test       : (100,)


## 5. Train/Val Split

In [ ]:
all_units = np.unique(unit_ids_full)
np.random.seed(SEED)
np.random.shuffle(all_units)

n_val       = int(len(all_units) * 0.20)
val_units   = set(all_units[:n_val])
train_units = set(all_units[n_val:])

assert len(val_units & train_units) == 0, '🚨 Data leakage !'

train_mask = np.array([uid in train_units for uid in unit_ids_full])
val_mask   = ~train_mask

X_train, y_train = X_train_full[train_mask], y_train_full[train_mask]
X_val,   y_val   = X_train_full[val_mask],   y_train_full[val_mask]

print(f'Moteurs train : {len(train_units)}  |  val : {len(val_units)}')
print(f'Séquences train : {X_train.shape[0]:,}  |  val : {X_val.shape[0]:,}')
print(' Split OK — seed=42')

Moteurs train : 80  |  val : 20
Séquences train : 12,641  |  val : 3,090
 Split OK — seed=42


## 6. Fonctions utilitaires

In [ ]:
def nasa_score(y_true, y_pred):
    d = np.array(y_pred) - np.array(y_true)
    return float(np.sum(np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1)))

def compute_metrics(y_true, y_pred, name=''):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    nasa = nasa_score(y_true, y_pred)
    bien = int(np.sum(np.abs(y_pred - y_true) <= 13))
    print(f'[{name}] RMSE:{rmse:.3f} MAE:{mae:.3f} R²:{r2:.4f} NASA:{nasa:.1f} Bien:{bien}/100')
    return {'model':name, 'rmse':rmse, 'mae':mae, 'r2':r2, 'nasa':nasa, 'bien_predits':bien}

def get_callbacks(model_name, model_dir):
    return [
        EarlyStopping(monitor='val_loss', patience=15,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=7, min_lr=1e-6, verbose=1),
        ModelCheckpoint(f'{model_dir}/{model_name}_best.keras',
                        monitor='val_loss', save_best_only=True, verbose=0),
        WandbMetricsLogger()
    ]

results = []
print(' Utilitaires définis')

 Utilitaires définis


## 7. CNN-LSTM Hybride


In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout, Bidirectional, SpatialDropout1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
import wandb


# 1. Seed
# ------------------------------
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)


# 2. Paramètres
# ------------------------------
WINDOW_SIZE = 50
N_FEATURES = X_train.shape[2]

DROPOUT_RATE = 0.1
BATCH_SIZE = 128
EPOCHS = 100
LR = 0.001


# 3. Wandb
# ------------------------------
wandb.init(
    project="industrial-ai-platform",
    name="CNN-BiLSTM-w50-Sweep1",
    config={
        "architecture": "CNN + BiLSTM (Sweep Best)",
        "conv_filters": [32, 64],
        "kernel_size": 3,
        "lstm_units": 64,
        "dropout": DROPOUT_RATE,
        "dense_units": 32,
        "learning_rate": LR
    }
)

# ------------------------------
# 4. Modèle same as stellar
# ------------------------------
model = Sequential([

    # CNN
    Conv1D(32, kernel_size=3, activation='relu', padding='same',
           input_shape=(WINDOW_SIZE, N_FEATURES)),
    SpatialDropout1D(DROPOUT_RATE),

    Conv1D(64, kernel_size=3, activation='relu', padding='same'),
    SpatialDropout1D(DROPOUT_RATE),

    # BiLSTM
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(DROPOUT_RATE),

    # Dense
    Dense(32, activation='relu'),
    Dense(1, activation='linear')

], name='CNN_BiLSTM_w50_Sweep1')

# ------------------------------
# 5. Compilation
# ------------------------------
model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='mse',
    metrics=['mae']
)

model.summary()

# ------------------------------
# 6. Callbacks (IMPORTANT)
# ------------------------------
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-5,
    verbose=1
)

# ------------------------------
# 7. Training
# ------------------------------
print('\n Training CNN-BiLSTM w50 Sweep 1...')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[lr_scheduler] + get_callbacks('cnn_bilstm_w50_sweep1', MODEL_DIR),
    verbose=1
)

# ------------------------------
# 8. Save
# ------------------------------
model.save(f'{MODEL_DIR}/cnn_bilstm_w50_sweep1.keras')

# ------------------------------
# 9. Evaluation
# ------------------------------
y_pred = model.predict(X_test, verbose=0).flatten()
m = compute_metrics(y_test, y_pred, 'CNN-BiLSTM-w50-Sweep1')
results.append(m)

# ------------------------------
# 10. Wandb log
# ------------------------------
wandb.log({
    'test_rmse': m['rmse'],
    'test_mae': m['mae'],
    'test_r2': m['r2'],
    'nasa_score': m['nasa'],
    'bien_predits': m['bien_predits']
})

wandb.finish()

print(f' Sweep 1 w50 model terminé — {len(history.history["loss"])} epochs')

Model: "CNN_BiLSTM_w50_Sweep1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 50, 32)         │         1,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 50, 32)         │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 50, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ (None, 50, 64)         │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 77,793 (303.88 KB)

 Trainable params: 77,793 (303.88 KB)

 Non-trainable params: 0 (0.00 B)


 Training CNN-BiLSTM w50 Sweep 1...
Epoch 1/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - loss: 4772.6523 - mae: 57.8770 - val_loss: 2216.4177 - val_mae: 40.3736 - learning_rate: 0.0010
Epoch 2/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 1804.1154 - mae: 37.3680 - val_loss: 1685.9594 - val_mae: 36.1897 - learning_rate: 0.0010
Epoch 3/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 1329.0563 - mae: 31.9611 - val_loss: 561.7582 - val_mae: 20.5327 - learning_rate: 0.0010
Epoch 4/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - loss: 495.6428 - mae: 18.6016 - val_loss: 299.6383 - val_mae: 14.1091 - learning_rate: 0.0010
Epoch 5/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 323.3761 - mae: 14.5773 - val_loss: 180.6847 - val_mae: 10.6598 - learning_rate: 0.0010
Epoch 6/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 220.6648 - mae: 11.6218 - val_loss: 136.5904 - val_mae: 9.1423 - learning_rate: 0.0010
Epoch 7/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 193.9860 - 

bien_predits,▁
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,███████████████████▄▄▃▃▃▂▂▂▂▁▁
epoch/loss,█▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/mae,█▅▄▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_loss,█▆▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_mae,█▇▄▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
nasa_score,▁
test_mae,▁
test_r2,▁
+1,...


 Sweep 1 w50 model terminé — 30 epochs
